### Acceptance Criteria
 - At least 3 JSON files land in /Volumes/dev_<you>/ops/landing/trips/, each a different slice of samples.nyctaxi.trips
 - The same data also exists as CSV and Parquet in sibling directories
 - One JSON file contains an extra column the others don't
 - One JSON file contains at least 5 deliberately bad records (nulls in key fields, negative fares)
 - You can list the files with dbutils.fs.ls and see distinct file sizes
### Subtasks
 - Export batch 1 — Read samples.nyctaxi.trips, limit() a few thousand rows, write JSON to the volume. Hint: .write.format("json").save(path) writes a directory of part-files; that's fine and realistic.
 - Export batches 2 and 3 — Use a different WHERE filter each time so the batches are distinct. Hint: filter on tpep_pickup_datetime ranges.
 - Create the schema-drift file — Add a computed column (e.g. fare_per_mile) before writing batch 4. This is LAB-03's evolution trigger.
 - Create the dirty file — union some rows with nulls and negatives. Hint: build them with spark.createDataFrame — remember the 128 MB local row-size cap on serverless.
 - Repeat for CSV and Parquet — Different directories. Hint: note which formats infer types and which don't; that's the LAB-03 lesson.
 - Verify — dbutils.fs.ls each directory.
#### Note
Keep this notebook. Later labs will re-run it to simulate new data arriving.

# Batches
1. tpep_pickup_date >= 2016-01-01 and tpep_pickup_date <= 2016-01-04 - 1624 records  
2. tpep_pickup_date >= 2016-01-05 and tpep_pickup_date <= 2016-01-08 - 1160 records  
3. tpep_pickup_date >= 2016-01-09 and tpep_pickup_date <= 2016-01-12 - 1460 records  
4. tpep_pickup_date >= 2016-01-13 and tpep_pickup_date <= 2016-01-15 - 787 records  
    a. Schema Drift Batch
5. tpep_pickup_date >= 2016-01-16 and tpep_pickup_date <= 2016-01-16 - limit 50 records  
    a. Dirty File - these 50 records will be unioned with dirty records having null and negative values

In [0]:
from pyspark.sql.functions import col

# JSON, CSV, Parquet Formats

In [0]:
formats: list = ["json", "csv", "parquet"]
etl_configuration: dict = {
    "1": {
        "low_watermark": "2016-01-01",
        "high_watermark": "2016-01-04",
        "special_case": None
    },
    "2": {
        "low_watermark": "2016-01-05",
        "high_watermark": "2016-01-08",
        "special_case": None
    },
    "3": {
        "low_watermark": "2016-01-09",
        "high_watermark": "2016-01-12",
        "special_case": None
    },
    "4": {
        "low_watermark": "2016-01-13",
        "high_watermark": "2016-01-15",
        "special_case": "schema_drift"
    },
    "5": {
        "low_watermark": "2016-01-16",
        "high_watermark": "2016-01-16",
        "special_case": "dirty_files"
    },
}

In [0]:
for format_type in formats:
    for batch_number, config in etl_configuration.items():
        print(f"###### Starting Batch {batch_number}! ######\n")
        print(f"Config: \n  {config} ")
        if config.get("special_case") is None:
            (
                spark.read.table("samples.nyctaxi.trips").select("*")
                .filter( 
                    "tpep_pickup_date >= cast(config.get("low_watermark", None) as date) and tpep_pickup_date <= cast(config.get("high_watermark", None) as date)" 
                )
                .write.format(format_type).mode("overwrite")
                .path("/Volumes/dev_ppetkov1/ops/landing/nyctaxi_trips_json")
            )
        elif config.get("special_case", None) == "schema_drift":
            # TODO: Implement schema drift
            pass
        elif config.get("special_case", None) == "dirty_files":
            # TODO: Implement dirty files
            pass

In [0]:
# Batch 1: 1624 records
(
    spark.read.table("samples.nyctaxi.trips").select("*")
    .filter( 
        "tpep_pickup_date >= cast('2016-01-01' as date) and tpep_pickup_date <= cast('2016-01-04' as date)" 
    )
    .write.format("json").mode("overwrite")
    .path("/Volumes/dev_ppetkov1/ops/landing/nyctaxi_trips_json")
)

In [0]:
# Batch 1: 1160 records
(
    spark.read.table("samples.nyctaxi.trips").select("*")
    .filter( 
        "tpep_pickup_date >= cast('2016-01-05' as date) and tpep_pickup_date <= cast('2016-01-08' as date)" 
    )
    .write.format("json").mode("overwrite")
    .path("/Volumes/dev_ppetkov1/ops/landing/nyctaxi_trips_json")
)

In [0]:
# Batch 1: 1460 records
(
    spark.read.table("samples.nyctaxi.trips").select("*")
    .filter( 
        "tpep_pickup_date >= cast('2016-01-09' as date) and tpep_pickup_date <= cast('2016-01-13' as date)" 
    )
    .write.format("json").mode("overwrite")
    .path("/Volumes/dev_ppetkov1/ops/landing/nyctaxi_trips_json")
)

In [0]:
(
    spark.read.table("samples.nyctaxi.trips").select("*")
    .filter( 
        "tpep_pickup_date >= cast('2016-01-14' as date) and tpep_pickup_date <= cast('2016-01-17' as date)" 
    ).withColumn("fare_per_mile", col("fare_amount") / col("trip_distance"))
    .write.format("json").mode("overwrite")
    .path("/Volumes/dev_ppetkov1/ops/landing/nyctaxi_trips_json")
)